# 02 · Train, đối chứng và lượng tử hóa

4 kiến trúc × 3 seed; dữ liệu test giữ độc lập.

## Mở trên Colab hoặc Jupyter

Đây là notebook hướng dẫn tái lập, các cell chưa được đánh dấu là đã thực thi. Kết quả huấn luyện thật có sẵn trong `artifacts/`.

**Colab:** tải ZIP đầy đủ lên Files, giải nén vào `/content/` để có `/content/Neural_Observatory`. Có thể dùng `import zipfile; zipfile.ZipFile('/content/Neural_Observatory.zip').extractall('/content')` với ZIP được bàn giao. Chọn CPU; bật T4 không tăng tốc NumPy tự động. Nếu cần dependency, chạy `%pip install -r /content/Neural_Observatory/requirements.txt`.

**Jupyter:** mở notebook từ `notebooks/` sau khi cài `requirements.txt`. Chạy các cell từ trên xuống.


In [ ]:
from pathlib import Path
import os, sys, subprocess
# Local: launch notebook from the extracted project or its notebooks/ folder.
# Colab: extract Neural_Observatory.zip into /content first (see instructions).
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/Neural_Observatory')]
project = next((p for p in candidates if (p/'observatory/engine.py').exists()), None)
assert project is not None, 'Extract the full project ZIP first and set project to its root.'
os.chdir(project)
sys.path.insert(0, str(project))
print('Project:', project)


## Protocol

55.000 train, 5.000 validation, 10.000 test. SGD momentum 0,9; batch 128; 18 epoch. Chọn checkpoint bằng validation loss. Đọc protocol trước khi xem test. Full training dùng NumPy CPU, không autograd.

In [ ]:
import json
protocol=json.loads(Path('artifacts/protocol.json').read_text())
protocol['protocol']

In [ ]:
RETRAIN = False  # Change to True to rerun every model.
if RETRAIN:
    subprocess.run([sys.executable,'main.py','train','--epochs','18','--seeds','42','43','44'],check=True)
else:
    print('Using the completed measured experiment provided in artifacts/.')

## Quantization tự viết

Một scale mỗi output channel. Trọng số chuyển sang signed byte; bias/scale float32. Đây là weight-only quantization. Đừng suy ra tốc độ hay RAM toàn chương trình từ số bit.

In [ ]:
from observatory.packing import quantize_row, load_packed, packed_forward
weights=[-0.91,-0.2,0.,0.11,0.72]
q,s=quantize_row(weights)
print('scale=',s,'q=',q,'reconstructed=',[v*s for v in q])

## Kết quả 24 bản

Mean ± SD là biến thiên ba seed. Hãy tìm ảnh đổi dự đoán và trường hợp INT8 chậm hơn.

In [ ]:
import csv
rows=list(csv.DictReader(open('artifacts/benchmark.csv')))
for row in rows:
    print(row['model'],row['seed'],row['variant'],row['test_accuracy'],row['model_bytes'],row['latency_p50_ms'])

## Câu hỏi nghiên cứu

Với ngân sách 64 KiB, mạng 64–32 INT8 hay mạng 20 FP32 đạt điểm validation tốt hơn? Tại sao giảm dung lượng không đủ để khẳng định tăng tốc?